# Lab 07 · Observability, tracing, and swappable agents

**Prepared by Mustafa Assaf** · Lecture 3: Debate, trace, decide · 15–20 minute lab

## Observability, tracing, and swappable agents

**So far:** in Lab 06, the chair kept a transcript in a Python list. You could read it after the run, but you could not watch the council while it worked, follow a decision back to its evidence in one step, or add an architect without touching the chair's code.

**New:** three properties that a multi-agent system needs before anyone can trust it or change it. Each one comes from one small mechanism:

- **Observable:** every message passes through a *message bus* that writes it to the **logs**: a live log stream that you watch while the council runs, and a log file with one JSON record per message that anyone can read, search, and count afterwards. With a free LangSmith key, the same run also appears in **LangSmith** as a trace: a tree of steps with their inputs, outputs, and timings.
- **Traceable:** every message has an id and cites what it is based on: sources, or earlier messages. Following those links in the log file answers the question "who said what, based on what?", even after the program has stopped.
- **Swappable:** the council finds its agents in a *registry* and talks to each one through the same interface. Adding or replacing an agent takes one line, and the council's code does not change.

All three rest on a fourth property: **structured** messages. Every message is a typed record with fixed fields, so code can check it, log it, and follow its links. The case is Lab 06's ShopEasy support search. Every message also carries one `trace_id`, like the correlation ID propagated across microservice hops in slides 46 and 48. This is an in-process agent log, not a deployed service trace.

**Optional: see the run in LangSmith.** [LangSmith](https://smith.langchain.com/) is a hosted tracing tool for LLM applications. Create a free account and an API key, save the key in Colab Secrets as `LANGSMITH_API_KEY`, and run this in a new cell right after the setup cell, before Step 1:

```python
import os
from google.colab import userdata
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
```

Without it, nothing is sent anywhere, and the lab runs the same, on the logs alone.

**At a glance**

![Lab 07 at a glance: Minimalist (model agent) or Modernizer (model agent) or Rulebook (added in one line), then Schema check (typed Message), then Message bus (publish + trace_id), then Live log (time · level · line) or council.log (one JSON per line) or LangSmith (trace tree (optional)), then Chair (majority or escalate), then Evidence trail (trace() from council.log).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/07.svg)

## Review your microservices lectures

- **Microservices & Their Patterns**, slides 46, 48.
- **Microservices Lecture 2 - Condensed DDD**, slides 30, 31.

**Recall:** Correlation IDs across service boundaries; Structured logs and immutable event facts.

Before running the example:

1. What is the difference between a trace ID and a message ID?
2. Why does an event or agent message need both identity and provenance?

Slide numbers count from the title slide, including appendices. The small scenarios and code are teaching adaptations, not production services. Use the lecture decks supplied in class; they are not uploaded with this course.

## The idea in context

Observability makes a workflow inspectable during and after a run. A message bus owns logging; a registry lets implementations share one interface.

Agent-run traces and microservice request traces share correlation principles but are different systems.

| Key term | Meaning |
| --- | --- |
| **Observability** | Seeing what a running system does from its outputs: logs, metrics, and traces. |
| **Trace** | A record of one run as a tree of steps, with each step's input, output, and timing; LangSmith shows traces of LLM applications. |
| **Log** | A time-stamped record of what happened; structured logs, such as JSON lines, can be searched and counted. |
| **Registry** | A lookup table of agents by name, so the system never hard-codes who takes part. |

## Pattern · Shared Memory

*Memory pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Registry & Adapter.*

| | |
| --- | --- |
| **Context** | Agents may work in separate contexts but still need a common source of truth for goals, progress, and handoffs. |
| **Problem** | Copying messages between agents duplicates context and creates inconsistent versions of the shared state. |
| **Solution** | Give the agents one shared memory, so that what one agent writes is immediately available to the others. |
| **Use when** | Agents share a project state; their own contexts should stay separate; a common task board is useful. |
| **Trade-off** | Reduces duplication, but concurrent updates need clear ownership to avoid overwrites and races. |
| **Flow** | Agent A → Shared memory → Agent B → Coordinated state |
| **Quality attributes** | Benefits: Interoperability, Reliability, Cost. Liabilities: Reliability, Security, Maintainability. |
| **In this lab** | The log is the shared memory: every message is written to it, and the live view and the tracer read it. Only the bus writes to the log, and a message cannot change once it is written, which addresses the pattern's main liability: concurrent writes cannot overwrite each other. The council's registry is the *Registry & Adapter* part. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'langsmith']):
    %pip install -q "agentic-ai-coursekit[framework] @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · A typed message

`Message` is a *dataclass*: a class that only holds named fields. `frozen=True` makes a message read-only once it is created, so nobody can change what an agent said after the fact. Every message has an `id`, a `sender`, a `kind`, an `option`, a one-sentence `claim`, and `cites`: the ids of the sources, or of the earlier messages, it is based on. These links are what make the council traceable. `dataclasses.asdict` turns a message into a dictionary, ready to be written to the log as JSON. `traceable`, from LangSmith's SDK, is used in Step 4.

A dataclass groups named fields in one record. `frozen=True` prevents changing a record after it is created.

In [ ]:
import collections, dataclasses, itertools, json, logging, sys
from langsmith import traceable  # traces each run for LangSmith; does nothing without a key

SOURCES = {"M27": "API composition fits small lookups by id; broad search can require large joins.",
           "M28": "CQRS supports rich queries but adds a stale read model.", "C1": "ShopEasy support searches orders and payments by name."}

@dataclasses.dataclass(frozen=True)
class Message:  # the typed schema: every message on the bus has exactly these fields
    id: str
    sender: str
    kind: str          # "proposal", "rejected", "decision", or "escalation"
    option: str
    claim: str
    cites: tuple = ()  # ids of sources, or of earlier messages
    trace_id: str = "support-review-1"  # one correlation id for this decision

## Step 2 · Logs: a live stream and a log file

Observability starts with logs. Python's `logging` module sends each *log record* to one or more *handlers*, and each handler formats and writes it somewhere. Our logger has two:

- `live` writes one readable line per message to the notebook, with the time and a *level*, as the council runs. This is the live view of the demo.
- `to_file` writes the same message to `council.log` as one JSON object per line, the format that log tools and dashboards search. `extra={"record": ...}` hands the message's JSON to that formatter.

`publish` is the bus: agents never print their own messages, they hand each one to `publish`, which logs it. A rejected message or an escalation is logged as a `WARNING`, so problems stand out in the live view and can be counted in the file. Nothing is printed yet.

In [ ]:
logger = logging.getLogger("council")
logger.setLevel(logging.INFO)
live = logging.StreamHandler(sys.stdout)  # the live view: one readable line per message, as it happens
live.setFormatter(logging.Formatter("%(asctime)s %(levelname)-7s %(message)s", "%H:%M:%S"))
to_file = logging.FileHandler("council.log", mode="w")  # the log file: one JSON object per line
to_file.setFormatter(logging.Formatter('{"time": "%(asctime)s", "level": "%(levelname)s", "message": %(record)s}'))
logger.handlers = [live, to_file]  # replaces the handlers of an earlier run of this cell
ids = itertools.count()  # message ids: m0, m1, ...

def publish(msg):  # the bus: every message is logged, as a warning when it needs attention
    level = logging.WARNING if msg.kind in ("rejected", "escalation") else logging.INFO
    text = f"{msg.trace_id} {msg.id:<4}{msg.sender:<11}{msg.kind:<11}{msg.option:<18}cites {list(msg.cites)}"
    logger.log(level, text, extra={"record": json.dumps(dataclasses.asdict(msg))})
    return msg

## Step 3 · Agents behind one interface

For this council, an agent is any function that takes a case and returns a reply with an option, a claim, and the ids it cites. `model_architect` builds such a function around a model call, and passes the sources as JSON, so the model sees each id exactly: it returns the inner function `propose`, which remembers its stance and its practice reply. `council` is the *registry*: a dictionary from each agent's name to its function. The council's code finds its agents only there, so it never needs to know how an agent works inside.

In [ ]:
def model_architect(stance, example):  # an architect driven by a model; any function from case to reply can join
    def propose(case):
        return json.loads(ask(f"You are an architect on a council. {stance}\nSources: {json.dumps(SOURCES)}\n"
                              f"Case: {json.dumps(case)}\nReply with JSON only, no code fences: "
                              '{"option": "API composition" or "CQRS", "claim": "one sentence", '
                              '"cites": ["source ids"]}', example=example))
    return propose

council = {  # the registry: the council's code finds its agents only here
    "Minimalist": model_architect("Prefer the simplest design that works.",
        '{"option": "API composition", "claim": "API composition avoids a new view database, but broad search may be costly.", "cites": ["M27"]}'),
    "Modernizer": model_architect("Prefer a read model that supports growing search traffic.",
        '{"option": "CQRS", "claim": "A separate read model supports the requested search.", "cites": ["M28", "C1"]}')}
OPTIONS, CASE = ("API composition", "CQRS"), {"name": "ShopEasy support", "cross_service_search": True}

## Step 4 · Run the council, and watch the logs and the trace

`run_council` asks every registered agent and checks each reply against the schema: a known option, and citations of real sources only. It publishes the reply as a `proposal`, or as a `rejected` warning. Then the chair publishes its own message: a `decision` if a strict majority of the registered architects agree with valid citations, or otherwise an `escalation` warning for the human. The chair's message cites the proposals it is based on. The log lines appear as each message is published. The last line printed is the chair's message as the log file recorded it.

`@traceable` turns `run_council` into a *trace* in LangSmith, and `traceable(name=name)(propose)` wraps each agent's call, so every agent, whatever it is inside, becomes one step of that trace. The logs record *what was said*; the trace records *how the run went*: which step called which, with what input and output, and how long each took.

In live mode, a model may cite `"[M28]"` with brackets instead of `"M28"`, or invent `"Case"` as a source ID. The citation check rejects those IDs and the log records a warning. This checks the ID, not the truth of the explanation.

The chair asks each registered function for a proposal. It counts only proposals with a recognized option and citation IDs, then requires a majority of the whole registry.

In [ ]:
@traceable(name="council")
def run_council(case):  # ask every registered agent, check each reply against the schema, then decide
    accepted = []
    for name, propose in council.items():
        r = traceable(name=name)(propose)(case)  # each agent is a step in the trace
        if not isinstance(r, dict):
            r = {}
        ok = r.get("option") in OPTIONS and bool(r.get("cites")) and set(r["cites"]) <= set(SOURCES)
        msg = publish(Message(f"m{next(ids)}", name, "proposal" if ok else "rejected", str(r.get("option")),
                              str(r.get("claim")), tuple(r.get("cites", ()))))
        if ok:
            accepted.append(msg)
    option, count = (collections.Counter(m.option for m in accepted).most_common(1) or [("none", 0)])[0]
    decided = 2 * count > len(council)  # a strict majority of the accepted proposals
    return publish(Message(f"m{next(ids)}", "Chair", "decision" if decided else "escalation",
                           option if decided else "no majority", f"{count} of {len(accepted)} accepted votes",
                           tuple(m.id for m in accepted if m.option == option or not decided)))
run_council(CASE)
print(open("council.log").readlines()[-1].strip())  # the same message, as the log file recorded it

## Step 5 · Trace a decision from the log, then add an agent

`trace` reads `council.log`, finds a message by its id, and follows its citations back until it reaches the sources. It needs nothing but the log file, so the same trace works after the program has stopped, or on another computer. Then a third agent joins: `rulebook` is Lab 00's rule, with no model at all. It has the same interface as the model architects, so adding it takes one line, and `run_council` runs unchanged with three agents.

The log stores message IDs and citation IDs. The `SOURCES` dictionary supplies the source text when the trace reaches a source ID; save that dictionary too if you need to reconstruct the trace later.

In [ ]:
def trace(msg_id, depth=0):  # follows message IDs in the log and looks up source text in SOURCES
    msg = {r["message"]["id"]: r["message"] for r in map(json.loads, open("council.log"))}[msg_id]
    print("    " * depth + f"{msg['id']} {msg['sender']}: {msg['option']}. {msg['claim']}")
    for ref in msg["cites"]:
        if ref in SOURCES:
            print("    " * (depth + 1) + f"{ref}: {SOURCES[ref]}")
        else:
            trace(ref, depth + 1)

def rulebook(case):  # A pattern-selection heuristic: no model, but the same interface
    search = case["cross_service_search"]
    return {"option": "CQRS" if search else "API composition", "claim": "Search suggests CQRS; review staleness and cost.", "cites": ["M28", "C1"]}

council["Rulebook"] = rulebook  # adding an agent is one line; run_council does not change
trace(run_council(CASE).id)

## What you should see

First, two architects disagree about API composition and CQRS, so the chair escalates. The log contains each proposal and the chair's outcome as JSON, all with `trace_id="support-review-1"`. Add Rulebook through the registry and run again: the CQRS proposal now wins 2–1. The trace follows message citations back to M28 and C1.

As in the microservices slides, a correlation ID groups related records. Message IDs identify individual records. These IDs have different jobs. This lab simulates an architecture review locally; it does not implement Kafka or distributed tracing across deployed services.

## Try it

1. Cite `M99` in one practice reply. Is the proposal rejected and recorded as a warning?
2. Read `council.log`, filter by `trace_id`, and count messages by sender and level.
3. Make the decision cite dissenting proposals too. What information does its trace gain?
4. Change the trace ID for a second support request. Can you separate both requests' records?
5. Add an architect that favours simpler operations. Which registry line changes, and does the chair change?
6. With a LangSmith key, compare the timed agent trace with the JSON log. Which question does each answer?

## Takeaway

A registry makes agents replaceable, structured logs show what happened, and IDs connect an outcome to its evidence. The same observability principle helps diagnose a microservice request crossing Orders, Inventory, and Payments. A source ID still needs a human check that the cited source supports the claim.